# NASA C-MAPSS FD001 forecasting (Kaggle)

Runs the committed C0/C1 engine-disjoint protocol only. FD001 is simulator-generated data; the notebook uses complete `train_FD001.txt` trajectories and does not extrapolate official truncated test trajectories.

In [ ]:
!pip -q uninstall -y jax jaxlib
!pip -q install tensorflow==2.18.0 keras==3.15.1 numpy==2.0.2 pandas==2.2.2 scikit-learn==1.9.0
!python -c "import sys,tensorflow as tf,sklearn,numpy,pandas; print({'python':sys.version.split()[0], 'tensorflow':tf.__version__, 'sklearn':sklearn.__version__, 'numpy':numpy.__version__, 'pandas':pandas.__version__, 'devices':[str(x) for x in tf.config.list_physical_devices()]})"

In [ ]:
from pathlib import Path
import hashlib, json, shutil
INPUT = Path('/kaggle/input')
archives = list(INPUT.rglob('kaggle_cmapss_fd001_forecasting_input_bundle.zip'))
assert len(archives) == 1, f'Attach exactly one Dataset containing kaggle_cmapss_fd001_forecasting_input_bundle.zip; found {archives}'
UNPACKED = Path('/kaggle/working/cmapss_forecasting_input')
if UNPACKED.exists(): shutil.rmtree(UNPACKED)
shutil.unpack_archive(archives[0], UNPACKED)
candidates = [p for p in UNPACKED.rglob('cmapss_fd001_forecasting.py') if p.parent.name == 'ieee_experiments' and p.parent.parent.name == 'research_v2']
assert len(candidates) == 1, f'Bundle must contain exactly one forecasting script; found {candidates}'
SOURCE_ROOT = candidates[0].parents[2]
WORK = Path('/kaggle/working/autosense_cmapss_fd001')
if WORK.exists(): shutil.rmtree(WORK)
shutil.copytree(SOURCE_ROOT, WORK)
expected = {
 'research_v2/ieee_experiments/cmapss_fd001_forecasting.py':'cee828114ca0e22800b6bf12faa9643db4646f0c764bbef81558e1a6243a3a65',
 'research_v2/CODEX_IEEE_EXECUTION_PLAN.md':'1e72e4109507e7968af365367d855fac495439173f38c4c7a72c4ece71c627bb',
 'research_v2/external_data/cmapss/train_FD001.txt':'963b5e22825b34d8b21c69e1aeb4af3e647050eb672ee8834ba4b5d91d2de0f8',
 'research_v2/ieee_results/cmapss_fd001_split.json':'9e4c0a4ab7cef63169c1bc87a52885af983ffab5ea8f0e8e886d352e63779003',
 'research_v2/ieee_results/cmapss_sensor_selection.csv':'5033e1390448a71d6ff29364489b7a122d7b9190ffc851d301168349d4c89665'}
actual = {rel: hashlib.sha256((WORK / rel).read_bytes()).hexdigest() for rel in expected}
assert actual == expected, json.dumps({'expected':expected,'actual':actual}, indent=2)
split = json.loads((WORK / 'research_v2/ieee_results/cmapss_fd001_split.json').read_text())
assert [len(split[k]) for k in ('train_engine_ids','validation_engine_ids','test_engine_ids')] == [70,15,15]
assert len(set(split['train_engine_ids']) | set(split['validation_engine_ids']) | set(split['test_engine_ids'])) == 100
selection = __import__('pandas').read_csv(WORK / 'research_v2/ieee_results/cmapss_sensor_selection.csv')
assert set(selection.loc[selection['Selected top four'].astype(str).str.lower().eq('true'),'Sensor']) == {'sensor_9','sensor_11','sensor_14','sensor_12'}
print(f'Validated exact NASA FD001 and pre-registered split/sensor hashes; writable root: {WORK}')

In [ ]:
%cd /kaggle/working/autosense_cmapss_fd001
!python -u research_v2/ieee_experiments/cmapss_fd001_forecasting.py

In [ ]:
import json, platform, sys, zipfile
import numpy as np, pandas as pd, tensorflow as tf
OUT = WORK / 'research_v2/ieee_results'
metrics = pd.read_csv(OUT / 'cmapss_fd001_metrics.csv')
horizon = pd.read_csv(OUT / 'cmapss_fd001_horizon_metrics.csv')
summary = pd.read_csv(OUT / 'cmapss_multiseed_summary.csv')
predictions = sorted((OUT / 'cmapss_fd001_predictions').glob('*.npz'))
assert len(metrics) == 55, f'Expected 55 detailed+macro metric rows, got {len(metrics)}'
assert len(horizon) == 550, f'Expected 550 detailed+macro horizon rows, got {len(horizon)}'
assert len(summary) == 8 and len(predictions) == 44
detailed = metrics[metrics['Target sensor'].ne('MACRO_AVERAGE_NORMALIZED')].copy()
assert len(detailed) == 44 and not detailed.duplicated(['Configuration','Target sensor','Model','Seed']).any()
assert set(detailed['Configuration']) == {'C0','C1'} and set(detailed['Target sensor']) == {'sensor_9','sensor_11','sensor_14','sensor_12'}
assert set(detailed['Seed']) == {11,42,84}
assert detailed['Lookback cycles'].eq(30).all() and detailed['Forecast horizon cycles'].eq(10).all()
assert detailed['Train engines'].eq(70).all() and detailed['Validation engines'].eq(15).all() and detailed['Test engines'].eq(15).all()
assert detailed['Train-only scaling'].eq(True).all() and detailed['Dataset SHA256'].eq(expected['research_v2/external_data/cmapss/train_FD001.txt']).all()
assert all({'unit_ids','target_start_cycles','truth','prediction','dataset_sha256','experiment_script_sha256','protocol_sha256','model_parameters','epochs_run','best_epoch','training_seconds'} <= set(np.load(p, allow_pickle=False).files) for p in predictions)
manifest = {'phase':'cmapss_fd001_forecasting','input_sha256':actual,'python':sys.version,'tensorflow':tf.__version__,'platform':platform.platform(),'devices':[str(x) for x in tf.config.list_physical_devices()],'expected_metric_rows':55,'expected_horizon_rows':550,'expected_prediction_files':44,'validation':'fixed 70/15/15 engine split; fixed top four sensors; C0/C1; no duplicate config/sensor/model/seed rows'}
manifest_path = OUT / 'kaggle_cmapss_fd001_forecasting_run_manifest.json'; manifest_path.write_text(json.dumps(manifest, indent=2))
zip_path = Path('/kaggle/working/cmapss_fd001_forecasting_results.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in [OUT/'cmapss_fd001_metrics.csv',OUT/'cmapss_fd001_horizon_metrics.csv',OUT/'cmapss_multiseed_summary.csv',OUT/'cmapss_fd001_split.json',OUT/'cmapss_sensor_selection.csv',manifest_path,*predictions]: z.write(p, p.relative_to(OUT))
print(detailed.groupby(['Configuration','Model'])[['MAE','RMSE','R2']].mean().round(6).to_string())
print(f'Validated 55 metrics, 550 horizon metrics, and 44 prediction files. Download: {zip_path}')